# Run-book 102-A11 — the rename to `rise-eo`, on a fresh install, on AML and locally

**What this proves:** gate 4 for the rename PR (spec 102 Amendment A11, #190). After `fsd` became
`rise-eo` / `rise_eo`, (1) a fresh install from this branch gives a working `rise-eo` command and
`import rise_eo`, and no `fsd` of either kind; (2) the AML images rebuild with the renamed package;
(3) one real AML run (download, training data, inference) on grid cell `s2grid=4772924` works with
the new job and STAC names; (4) one local run works; (5) the DemoRF bundle already in the registry
is refused by `load` with the A11.5 message, and works once its adapter's import is changed and the
bundle saved again.

**Claude does not run this** (spec 24). You run it cell by cell and paste back the **Summary**
cell's output, or a failing cell's message. Every check is a plain `assert` you can read: the
text above a check cell says what PASS means, and the `assert` says the same thing in code.

**Needs:** your Azure login (`az login`) with access to the workspace and blob; your config
(`rise-eo init`, including `image_registry` and `model_registry`); network. **If this is the first
time you use the renamed package**, the config still sits in the old folder: the first cell that
reads it fails with the exact `mv` command to run. Run it, then re-run the cell.
**Time (estimates):** Step 1 about 5-10 min (a `pip install`); Step 2 about 10-20 min per image that
has to be built, nothing if reused; Step 3 about 30-45 min (three AML jobs, each with node start);
Steps 4-5 about 10 min. **Data:** one month of Sentinel-2 L2A over one grid cell, bands B04, B08 and
SCL, about 2 GB (estimate).

## Before you start

**Kernel — which Python runs the cells.** Use **`.venv`**: the Python at `fsd/.venv/bin/python`
in the **main** checkout (Python 3.11). It is the only environment with rise-eo's dependencies.
(The folder is still called `fsd/` until the repo is transferred and renamed.)
- Open VS Code on the **`fsd/` folder** (the main checkout) — even when this notebook lives in a
  worktree under `fsd/.claude/worktrees/<name>/`. Then **Select Kernel** (top right) →
  **Python Environments** → **`.venv`**.
- Not listed? **Select Another Kernel… → Python Environments → Enter interpreter path…** and give
  the main checkout's `fsd/.venv/bin/python`.
- The **Setup** cell below checks this and stops with a message if the kernel is wrong.

**One-time install, in a terminal, before Setup.** The main `.venv` still has the old `fsd` distribution,
and `import rise_eo` reads the metadata of `rise-eo`, so Setup fails until you swap them. Replace
`<main>` with the main checkout (the folder that holds `.venv`) and `<this checkout>` with the folder
that holds this notebook's `runbooks/`:

```bash
<main>/.venv/bin/pip uninstall -y fsd
<main>/.venv/bin/pip install -e "<this checkout>[dev,local,s3,notebooks,grid,azure,aml,mpc,titiler,serving]"
```

This makes the `.venv` an editable install of this branch. After the PR merges, run the second command
again from the main checkout (`<main>` in place of `<this checkout>`) so the install points back at it.
Step 1 below still installs into a fresh environment of its own.

**Stop / resume.** Interrupt the kernel (■) to stop. Re-running a cell resumes: downloads skip files
that are already complete, an image that is built is reused. After a kernel restart, run **Setup** and
**Settings** again, then continue at any step — every step reads what it needs from disk.

**When you are done.** Paste back the **Summary** output (plus any failing cell's message). Then
**Restart & Clear All Outputs** and save: this file is tracked in git, and a test refuses saved outputs.

In [ ]:
# Setup -- which Python, and which rise-eo. Nothing to change here.
import importlib.metadata
import sys
from pathlib import Path

# 1. Which Python? It must be rise-eo's own virtual environment, fsd/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not rise-eo's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the checkout that owns the venv

# 2. Which rise-eo? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "rise_eo").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
try:
    importlib.metadata.version("rise-eo")
except importlib.metadata.PackageNotFoundError:
    raise AssertionError(
        "The .venv has no `rise-eo` distribution (it still has `fsd`). Run the one-time install "
        "under 'Before you start' in a terminal, restart the kernel, then run this cell again.") from None
import rise_eo

assert Path(rise_eo.__file__).is_relative_to(CHECKOUT / "src"), (
    f"rise_eo was already imported from {rise_eo.__file__}. Restart the kernel, then run this cell first.")
print("Python  :", sys.executable)
print("rise_eo :", rise_eo.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else. The Azure coordinates (subscription,
resource group, workspace, cluster, managed identity, the two registries) are read from your config by
`rise_eo.config.load()`; this file holds none of them. **Replace `<your-user>`** in `AZ_ROOT` with your
own blob prefix. `OLD_MODEL_REF` is the DemoRF version in your model registry that was saved before the
rename (Step 5), in the form `name:N`.

In [ ]:
# Settings
import datetime
import json

from rise_eo.storage import fs
from rise_eo.storage.azure import configure_storage

cfg = rise_eo.config.load()      # subscription, resource group, workspace, cluster, uami, registries
assert cfg.image_registry, "set image_registry: `rise-eo init --set image_registry=abfss://.../image_registry`"
assert cfg.model_registry, "set model_registry: `rise-eo init --set model_registry=abfss://.../model_registry`"
configure_storage("azure")       # so every step can read blob, also after a kernel restart

AZ_ROOT = "abfss://data@strisewesteurope.dfs.core.windows.net/<your-user>/a11-rename"
assert "<" not in AZ_ROOT, "set your own blob prefix in AZ_ROOT (replace <your-user>)"

OUT = MAIN_CHECKOUT / "tests" / "outputs" / "102_a11_rename"       # local, gitignored; outlives a worktree
FRESH_VENV = OUT / "fresh-venv"                                     # Step 1 installs into this
FRESH_EXTRAS = "local,azure,mpc,grid"                               # what a user of the AML path installs

SHAPEFILES = CHECKOUT / "notebooks" / "shapefiles"
ROI = SHAPEFILES / "s2grid=4772924.geojson"                         # one 5 km cell inside tile T33UWP
LABELS = SHAPEFILES / "AT_2018_TRAIN.geojson"                       # 900 fields: fid, crop
COLLECTION = "sentinel-2-l2a"
START, END = datetime.datetime(2018, 6, 1), datetime.datetime(2018, 7, 1)   # one month: about 2 GB
MOSAIC_DAYS = 10                                                    # T = ceil(30 / 10) = 3
BANDS = ["B04", "B08", "SCL"]                                       # SCL is the mask, consumed not returned

ARCHIVE = f"{AZ_ROOT}/imagery"                                      # its own archive: nothing skipped as present
CATALOG = rise_eo.archive_catalog_filepath(ARCHIVE, COLLECTION)
RUN_ROOT = f"{AZ_ROOT}/run"                                         # AML shard CSVs and _status/ JSON land here

MODEL_NAME = "a11-rename-rf"                                        # the model this run deploys (Step 3)
OLD_MODEL_REF = "crop-rf:1"                                         # EDIT: the pre-rename DemoRF version, name:N
RESAVED_NAME = "a11-rename-demorf"                                  # the name Step 5 saves the repaired bundle under

# The adapter this run trains for: NDVI over B04/B08, the cloud mask applied first. Written to disk
# because a bundle embeds the adapter's SOURCE FILE.
ADAPTER_SRC = """\
from rise_eo.bands import modify
from rise_eo.model.adapter import BaseModelAdapter


class A11RF(BaseModelAdapter):
    required_bands = ["B04", "B08"]
    n_timestamps = 0                       # set per instance at bundle time
    output_dtype = "uint8"
    output_nodata = 255
    output_band_names = ["crop_class"]
    feature_sequence = [
        (modify.mask_invalid_and_interpolate, {}),
        (modify.compute_bands, dict(bands_to_compute=["NDVI"])),
        (modify.remove_bands, dict(bands_to_remove=["B04", "B08"])),
    ]

    def load(self):
        import joblib
        self.clf, self.le = joblib.load(self.artifacts["model"])

    def predict(self, X_chunk):
        return self.clf.predict(X_chunk).astype("uint8")
"""
OUT.mkdir(parents=True, exist_ok=True)


def aml_kwargs(env_key):
    """`runner_kwargs` for an AML verb, with the image Step 2 recorded on disk ("aml_env" or "infer_env")."""
    envs = json.loads((OUT / "envs.json").read_text())
    return dict(
        cluster=cfg.cluster,
        environment=envs[env_key]["ref"],
        root=RUN_ROOT,
        identity_client_id=cfg.uami_client_id,       # the NODES' managed identity, not your login
        subscription_id=cfg.subscription_id,
        resource_group_name=cfg.resource_group,
        workspace_name=cfg.workspace,
        poll_interval_seconds=10,
    )

## Step 1 — a fresh install from this branch (A11 verify, first bullet)

**What it does:** creates a new virtual environment under `OUT`, installs this checkout into it with
`pip install "<checkout>[...]"` (not editable, so what runs is what the wheel carries), then runs the
new command and the new import in that environment. Nothing from the main `.venv` is involved.

**PASS if:** `rise-eo --help` exits 0 and lists `init`; `python -c "import rise_eo"` prints a version
and a path inside the new environment; the old command `fsd` does not exist there; `import fsd` fails
there.
**Time / disk:** 5-10 min (estimate), about 1.3 GB on disk (estimate).

In [ ]:
# Step 1 -- install into a fresh venv. Skipped if it already holds rise-eo.
import subprocess

BIN = FRESH_VENV / "bin"     # POSIX layout; on Windows this is Scripts/


def sh(*args, check=True):
    """Run a command, return (returncode, stdout + stderr). Raises on failure unless check=False."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{args!r} exited {r.returncode}:\n{r.stdout}\n{r.stderr}")
    return r.returncode, r.stdout + r.stderr


if not (BIN / "rise-eo").exists():
    sh(sys.executable, "-m", "venv", FRESH_VENV)
    sh(BIN / "python", "-m", "pip", "install", f"{CHECKOUT}[{FRESH_EXTRAS}]")
print("fresh venv at", FRESH_VENV)

In [ ]:
# Step 1 -- check it.
rc, help_text = sh(BIN / "rise-eo", "--help", check=False)
assert rc == 0 and "init" in help_text, f"`rise-eo --help` exited {rc}:\n{help_text}"

rc, where = sh(BIN / "python", "-I", "-c",
               "import rise_eo, pathlib; print(rise_eo.__version__, pathlib.Path(rise_eo.__file__).parent)",
               check=False)
assert rc == 0, f"import rise_eo failed in the fresh venv:\n{where}"
version, location = where.split(None, 1)
assert Path(location.strip()).is_relative_to(FRESH_VENV), f"rise_eo imported from {location!r}, not the fresh venv"

assert not (BIN / "fsd").exists(), "the old `fsd` command exists in the fresh venv"
rc, _ = sh(BIN / "python", "-I", "-c", "import fsd", check=False)
assert rc != 0, "`import fsd` still works in the fresh venv"
(OUT / "step1.json").write_text(json.dumps({"version": version, "location": location.strip()}))
print(f"PASS: rise-eo {version} installed at {location.strip()}; no fsd")

## Step 2 — rebuild the images with the renamed package (A11.5, image rows)

**What it does:** declares both node images from this checkout (`rise_eo="path:<this checkout>"`, so the
node runs this branch) and asks `rise_eo.aml.ensure_environment` for each: `rise-eo-aml-env` (download,
datacube builds) and `rise-eo-infer-sklearn` (inference). The wheel is `rise_eo-*.whl` and the node
commands are `python -m rise_eo.workflows...`. Every digest changes with the rename, so both images
build. `alias=None` (the default): no shared registry pointer such as `current` moves.

**PASS if:** both environments are reused, i.e. built. The first run **builds**: the cell then stops
with a link. Open it, wait for **Build status: Succeeded** (~10-20 min), and run the cell again; it
then says `reused`. The image registry's definition for each carries the key `rise_eo`.

In [ ]:
# Step 2 -- ensure both environments for THIS checkout.
import rise_eo.aml
from rise_eo.image import ImageDefinition
from rise_eo.image import registry as image_registry

BASE = ImageDefinition(name="rise-eo-aml-env", rise_eo=f"path:{CHECKOUT}", extras=("local", "azure", "mpc"))
INFER = BASE.derive(name="rise-eo-infer-sklearn", extra_pip=("scikit-learn", "joblib"))

envs = {}
for key, defn in (("aml_env", BASE), ("infer_env", INFER)):
    env = rise_eo.aml.ensure_environment(
        defn, registry=cfg.image_registry,
        resource_group=cfg.resource_group, workspace=cfg.workspace, storage="azure",
    )
    print(key, env.ref, "reused" if env.reused else "JUST BUILT")
    assert env.reused, (
        f"{env.ref} was just built: open {env.build_url} , wait for `Build status: Succeeded`, "
        "then re-run this cell.")
    resolved = image_registry.resolve(env.registry_ref, cfg.image_registry)
    assert "rise_eo" in resolved.definition and "fsd" not in resolved.definition, (
        f"{env.registry_ref}: image definition keys are {sorted(resolved.definition)}")
    envs[key] = {"ref": env.ref, "registry_ref": env.registry_ref}
(OUT / "envs.json").write_text(json.dumps(envs))
print("PASS:", envs)

## Step 3 — one real AML run on `s2grid=4772924` (A11.5: AML job names, STAC ids)

**What it does:** all three legs on AML with the Step 2 images. (a) `rise_eo.download` of one month of
B04, B08 and SCL over the cell (MPC). (b) `rise_eo.create_training_data` for the labelled fields inside
the cell; then, on this machine, a small random forest, bundled with the adapter above, verified on the
inference image (`verify_image`) and deployed under `MODEL_NAME`. (c) `rise_eo.run_inference` off the
deployed name over the same cell.

**PASS if:** the catalog holds granules; the training arrays have one row per labelled field; the
bundle's manifest still says `fsd_bundle_version` (the on-disk key is unchanged); `verify_image` passes;
`deploy` returns a `name:N` ref; inference writes at least one COG and a STAC collection whose id is
`rise-eo-inference` (not `fsd-inference`). The AML jobs are named `rise-eo-...` in Studio.
**Time:** ~30-45 min (estimate): download ~10, training data ~10, verify ~5, inference ~10.

In [ ]:
# Step 3a -- download (AML). Reads the Step 2 image from disk.
catalog_fp = rise_eo.download(
    roi=str(ROI), startdate=START, enddate=END, bands=BANDS, dst_folderpath=ARCHIVE,
    source="mpc", collection=COLLECTION, max_tiles=40, storage="azure",
    runner="aml", runner_kwargs=aml_kwargs("aml_env"),
)
print(catalog_fp)

In [ ]:
# Step 3a -- check it: the catalog holds granules, each with the three bands.
from rise_eo.catalog.catalog import TileCatalog

gdf = TileCatalog(CATALOG).read()
assert len(gdf) >= 1, f"expected >= 1 catalog row in {CATALOG}, found 0"
for _, r in gdf.iterrows():
    files = r["files"].split(",")
    assert all(f"{b}.tif" in files for b in BANDS), f"{r['id']}: files {files} lack one of {BANDS}"
print(f"PASS: {len(gdf)} granule(s) in the catalog")

In [ ]:
# Step 3b -- training data on AML. Labels are clipped to the cell; fields outside would only be
# chased once and recorded as known-empty, but that is preflight for no imagery.
import geopandas as gpd

cell = gpd.read_file(ROI)
fields = gpd.read_file(LABELS).to_crs(cell.crs)
fields = fields[fields.intersects(cell.geometry.iloc[0])]
print(f"{len(fields)} labelled fields inside {ROI.name}")

td = rise_eo.create_training_data(
    label_polygons=fields, catalog_filepath=CATALOG,
    startdate=START, enddate=END, mosaic_days=MOSAIC_DAYS, bands=BANDS,
    id_col="fid", label_col="crop", collection=COLLECTION,
    export_folderpath=str(OUT / "training_data"),
    aggregate="median_per_id",                    # one row per field
    overwrite=False,
    runner="aml", runner_kwargs=aml_kwargs("aml_env"),
    run_folderpath=f"{AZ_ROOT}/runs/train",
)
d = td.load()
assert "features" in d and len(d["feature_labels"]) >= 1, sorted(d)
print({k: getattr(v, "shape", len(v)) for k, v in d.items()})

In [ ]:
# Step 3b -- train, bundle, verify on the inference image, deploy.
# `td` comes from the cell above (after a restart re-run it: it skips what is already built).
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

from rise_eo.bands import modify
from rise_eo.model import bundle, verify_image
from rise_eo.model.features import apply_features

MODEL_DIR = OUT / "model"
MODEL_DIR.mkdir(exist_ok=True)
(MODEL_DIR / "a11_adapter.py").write_text(ADAPTER_SRC)
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))
import a11_adapter

d = td.load()                  # `td` is the previous cell's; after a restart re-run that cell first
raw, y = d["features"], np.asarray(d["feature_labels"])
band_indices = {b: i for i, b in enumerate(td.bands)}
feats5d, _ = apply_features(modify.expand_flattened(raw.astype(float)), dict(band_indices),
                            feature_sequence=a11_adapter.A11RF.feature_sequence)
X = np.squeeze(feats5d, axis=(2, 3)).reshape(len(y), -1)
keep = ~np.isnan(X).any(axis=1)
le = LabelEncoder()
clf = RandomForestClassifier(n_estimators=50, random_state=0).fit(X[keep], le.fit_transform(y[keep]))
model_fp = str(MODEL_DIR / "rf.joblib")
joblib.dump((clf, le), model_fp)

adapter = a11_adapter.A11RF()
adapter.n_timestamps = rise_eo.compute_n_timestamps(START, END, MOSAIC_DAYS)
adapter.artifacts = {"model": model_fp}
bundle_dir = bundle.save(
    adapter=adapter, artifacts={"model": model_fp}, dst=str(OUT / "bundle"),
    code=[str(MODEL_DIR / "a11_adapter.py")], requirements=["scikit-learn", "joblib"],
)
manifest = bundle.read_spec(bundle_dir)
assert "fsd_bundle_version" in manifest, "the on-disk key fsd_bundle_version must be unchanged"

envs = json.loads((OUT / "envs.json").read_text())
vres = verify_image(bundle_dir, environment=envs["infer_env"]["ref"], runner="aml",
                    runner_kwargs=aml_kwargs("infer_env"), registry=cfg.image_registry)
assert vres["pass"], vres
MODEL_REF = rise_eo.deploy(
    bundle_dir, name=MODEL_NAME, registry=cfg.model_registry, storage="azure",
    environment=envs["infer_env"]["ref"], verified=vres,
)
(OUT / "model_ref.json").write_text(json.dumps({"ref": MODEL_REF, "bundle_dir": str(bundle_dir)}))
print("PASS: deployed", MODEL_REF)

In [ ]:
# Step 3c -- inference on AML, off the deployed name.
MODEL_REF = json.loads((OUT / "model_ref.json").read_text())["ref"]
result = rise_eo.run_inference(
    model=MODEL_REF, registry=cfg.model_registry,
    output_folderpath=f"{RUN_ROOT}/model_outputs/{ROI.stem}",
    roi=str(ROI), catalog_filepath=CATALOG,
    startdate=START, enddate=END, mosaic_days=MOSAIC_DAYS, bands=["B04", "B08", "SCL"],
    collection=COLLECTION, merge="reproject", overwrite=False,
    storage="azure", runner="aml", runner_kwargs=aml_kwargs("infer_env"),
)
print(len(result.output_filepaths), result.merged_filepath, result.stac_catalog_filepath)

In [ ]:
# Step 3 -- check it: outputs exist, and the STAC collection carries the new default id.
import posixpath

assert len(result.output_filepaths) >= 1, "inference wrote no COG"
for fp in result.output_filepaths:
    assert fs.exists(fp), f"missing output {fp}"
stac_root = posixpath.dirname(result.stac_catalog_filepath)
collections = [p for p in fs.glob(f"{stac_root}/*/collection.json")]
assert any("/rise-eo-inference/" in p for p in collections), f"STAC collections: {collections}"
assert not any("/fsd-inference/" in p for p in collections), f"old collection id still written: {collections}"
print("PASS:", len(result.output_filepaths), "COG(s);", collections)

## Step 4 — one local run (A11: `runner="local"`)

**What it does:** the same two verbs with the default local runner, on the tracked tutorial fixture
`tests/data/tutorial/` (real Sentinel-2 pixels over the same cell, 27 MB, no network): training data
for its 43 labelled fields, then inference with the bundle Step 3 saved. Nothing is read from Azure.

**PASS if:** the training arrays have one row per field, and inference writes at least one COG and a
STAC collection with id `rise-eo-inference`.
**Time:** ~5 min (estimate; the fixture's field builds are the slow part).

In [ ]:
import os
os.chdir(CHECKOUT)  # the fixture catalog's image paths are relative to the repo root (#191)

# Step 4 -- local run on the tutorial fixture.
FIXTURE = CHECKOUT / "tests" / "data" / "tutorial"
FIX_CATALOG = str(FIXTURE / "sentinel-2-l2a" / "catalog.parquet")
LOCAL = OUT / "local"

td_local = rise_eo.create_training_data(
    label_polygons=str(FIXTURE / "fields.geojson"), catalog_filepath=FIX_CATALOG,
    startdate=START, enddate=END, mosaic_days=MOSAIC_DAYS, bands=BANDS,
    id_col="fid", label_col="label", aggregate="median_per_id",
    export_folderpath=str(LOCAL / "training_data"),
)
dl = td_local.load()
assert len(dl["feature_labels"]) >= 1, "no local training rows"

bundle_dir = json.loads((OUT / "model_ref.json").read_text())["bundle_dir"]
result_local = rise_eo.run_inference(
    model=bundle_dir, output_folderpath=str(LOCAL / "model_outputs"),
    roi=str(FIXTURE / "roi.geojson"), catalog_filepath=FIX_CATALOG,
    startdate=START, enddate=END, mosaic_days=MOSAIC_DAYS, bands=BANDS,
    collection=COLLECTION, overwrite=True, progress=False,
)
assert len(result_local.output_filepaths) >= 1, "local inference wrote no COG"
stac_dir = Path(result_local.stac_catalog_filepath).parent
assert (stac_dir / "rise-eo-inference" / "collection.json").exists(), sorted(p.name for p in stac_dir.iterdir())
print("PASS:", len(dl["feature_labels"]), "training rows;", len(result_local.output_filepaths), "COG(s)")

os.chdir(CHECKOUT / "runbooks")

## Step 5 — the DemoRF bundle already in the registry (A11.5, bundle row)

**What it does:** (a) downloads `OLD_MODEL_REF` (saved before the rename, its adapter imports `fsd`) to
`OUT` and tries to load it. (b) Follows the error message: changes `fsd` to `rise_eo` in the named file
under `code/`, loads the bundle again, saves it as a new bundle, verifies it on the inference image and
deploys it under `RESAVED_NAME`. The old registry version is not touched.

**PASS if:** (a) `bundle.load` raises `ValueError` whose message says the bundle was saved with
fsd <= v0.2 and names the file under `code/`; (b) after the edit `bundle.load` succeeds and `deploy`
returns a `name:N` ref.
**Time:** ~5-10 min (estimate), mostly `verify_image`'s node job.

In [ ]:
# Step 5a -- the old bundle must refuse to load, and say how to fix it.
from rise_eo.model import bundle
from rise_eo.model import registry as model_registry
from rise_eo.workflows.infer_shard import fetch_bundle_to_scratch

old = model_registry.resolve(OLD_MODEL_REF, cfg.model_registry)
OLD_LOCAL = OUT / "old_bundle"
fetch_bundle_to_scratch(old.path, str(OLD_LOCAL))

try:
    bundle.load(str(OLD_LOCAL))
except ValueError as exc:
    message = str(exc)
else:
    raise AssertionError(f"{OLD_MODEL_REF} loaded; it was expected to import fsd")
print(message)
assert "fsd <= v0.2" in message and "change `fsd` to `rise_eo`" in message and "code" in message, message
(OUT / "old_message.txt").write_text(message)
print("PASS: refused with the A11.5 message")

In [ ]:
# Step 5b -- do what the message says, then save the bundle again.
import re

from rise_eo.model import bundle, verify_image

envs = json.loads((OUT / "envs.json").read_text())

named = sorted(OLD_LOCAL.glob("code/**/*.py"))
edited = []
for path in named:
    text = path.read_text()
    new = re.sub(r"\bfrom fsd\b", "from rise_eo", text)
    new = re.sub(r"\bimport fsd\b", "import rise_eo", new)
    if new != text:
        path.write_text(new)
        edited.append(str(path.relative_to(OLD_LOCAL)))
assert edited, "no file under code/ imported fsd"
print("edited:", edited)

adapter_old = bundle.load(str(OLD_LOCAL))                    # loads now
RESAVED = OUT / "resaved_bundle"
bundle_dir = bundle.save(
    adapter=adapter_old, artifacts=adapter_old.artifacts, dst=str(RESAVED),
    code=[str(p) for p in named], requirements=bundle.read_spec(str(OLD_LOCAL)).get("requirements"),
)
vres = verify_image(bundle_dir, environment=envs["infer_env"]["ref"], runner="aml",
                    runner_kwargs=aml_kwargs("infer_env"), registry=cfg.image_registry)
assert vres["pass"], vres
RESAVED_REF = rise_eo.deploy(bundle_dir, name=RESAVED_NAME, registry=cfg.model_registry, storage="azure",
                             environment=envs["infer_env"]["ref"], verified=vres)
(OUT / "resaved_ref.json").write_text(json.dumps({"ref": RESAVED_REF, "edited": edited}))
print("PASS: saved again as", RESAVED_REF)

## Summary — paste this output back

Re-reads everything from disk, so it works after a restart.

In [ ]:
from rise_eo.catalog.catalog import TileCatalog

summary = {
    "step1_fresh_install": json.loads((OUT / "step1.json").read_text()),
    "step2_images": json.loads((OUT / "envs.json").read_text()),
    "step3_model": json.loads((OUT / "model_ref.json").read_text())["ref"],
    "step3_catalog_rows": len(TileCatalog(CATALOG).read()),
    "step4_local_cogs": sorted(p.name for p in (OUT / "local" / "model_outputs").rglob("*.tif"))[:5],
    "step5_old_ref": OLD_MODEL_REF,
    "step5_old_message": (OUT / "old_message.txt").read_text(),
    "step5_resaved": json.loads((OUT / "resaved_ref.json").read_text()),
}
print(json.dumps(summary, indent=2, default=str))